# Embedding archive analysis
Load original genotypic tensors independently of the experiment environment. The notebook chooses representations, projections and viewer formats. Images are optional.

Use the Python 3.13 kernel from the separate environment described in `../docs/embedding_analysis.md`. Empty `ARCHIVE_PATHS` runs a small synthetic example with mixed image availability; it never downloads a model.


In [ ]:
from pathlib import Path
import sys
# Support launching Jupyter from the repository root or notebooks directory.
REPO_ROOT = Path.cwd() if (Path.cwd() / "evolutionary_prompt_embedding").is_dir() else Path.cwd().parent
sys.path.insert(0, str(REPO_ROOT))
from IPython.display import display
from evolutionary_prompt_embedding.archive import EmbeddingArchiveReader
from evolutionary_prompt_embedding.analysis import (
    ProjectionConfig, available_representations, compute_projection,
    create_demo_archive, inspect_record, load_records,
)
from evolutionary_prompt_embedding.viewers import LocalViewerServer, show_atlas, show_3d, show_gallery


## Choose inputs and analysis settings
Selections here define the population used to fit a shared projection. Display filters later retain its coordinates.

Set `VIEWER` to `"atlas"`, `"3d"`, or `"both"`. Choose `"pca"`, `"umap"`, or `"tsne"`. t-SNE can be expensive on a full run; records are never silently sampled. Pooled-dependent representations are unavailable for token-only archives. Tiny datasets may require PCA and fewer dimensions.


In [ ]:
ARCHIVE_PATHS = []  # e.g. ["/absolute/path/to/embedding_runs/run-..."]
GENERATIONS = None  # e.g. [0, 10, 99]
ISLANDS = None  # e.g. [0, 1]; None includes all
METADATA_FILTERS = {}  # e.g. {"category": ["Abstract"]}
REPRESENTATION = "combined_append"  # token, pooled, combined_avg, combined_append
ALGORITHM = "umap"
VIEWER = "atlas"
REPROJECT = False
PROJECTION_PARAMETERS = {}  # e.g. {"n_neighbors": 15, "min_dist": 0.1} for UMAP


In [ ]:
archive_paths = ARCHIVE_PATHS or [create_demo_archive()]
archives = [EmbeddingArchiveReader(path) for path in archive_paths]
records = load_records(archives, generations=GENERATIONS, islands=ISLANDS, filters=METADATA_FILTERS)
print("Available representations:", available_representations(archives))
print("Committed records:", len(records))
print("Snapshots:", [a.manifest["snapshots"] for a in archives])
display(records.head())


## Compute or reuse projections
Native tensors are read in batches and converted to FP32 only for analysis. IncrementalPCA centres the representation without standardizing coordinates or normalizing vectors. UMAP/t-SNE operate on at most 64 PCA components. Neighbours use Euclidean distance in that PCA space, not the displayed 2D/3D geometry.


In [ ]:
if VIEWER not in ("atlas", "3d", "both"):
    raise ValueError("Choose atlas, 3d, or both")
dimensions = [2, 3] if VIEWER == "both" else [3 if VIEWER == "3d" else 2]
projections = {
    dim: compute_projection(
        archives, records,
        ProjectionConfig(representation=REPRESENTATION, algorithm=ALGORITHM,
                         dimensions=dim, seed=42, parameters=PROJECTION_PARAMETERS),
        force=REPROJECT,
    ) for dim in dimensions
}
print("Projection keys:", {dim: frame.projection_key.iloc[0] for dim, frame in projections.items()})


## Display filters and viewers
Filtering here keeps the fitted coordinates. Re-run the cells below after changing filters. Atlas supplies coordinated 2D exploration; the 3D view supplies orbit controls, point inspection and up to 2,000 thumbnail markers. Every point remains selectable.

The server serves only registered assets and images on loopback. The first 3D launch downloads a pinned, checksum-verified deck.gl bundle; later launches use the local asset. Atlas uses its installed widget bundle.


In [ ]:
DISPLAY_GENERATIONS = None
DISPLAY_ISLANDS = None
DISPLAY_MIN_FITNESS = None
DISPLAY_OBJECTIVE_MINIMA = {}  # e.g. {"objective_0": 0.5}

def display_filter(frame):
    selected = frame.copy()
    if DISPLAY_GENERATIONS is not None:
        selected = selected[selected.generation.isin(DISPLAY_GENERATIONS)]
    if DISPLAY_ISLANDS is not None:
        selected = selected[selected.island_id.isin(DISPLAY_ISLANDS)]
    if DISPLAY_MIN_FITNESS is not None:
        selected = selected[selected.fitness >= DISPLAY_MIN_FITNESS]
    for objective, minimum in DISPLAY_OBJECTIVE_MINIMA.items():
        selected = selected[selected[objective] >= minimum]
    return selected

visible = {dim: display_filter(frame) for dim, frame in projections.items()}
if any(frame.empty for frame in visible.values()):
    raise ValueError("No records match the display filters")
# Re-running this cell shuts down the previous server.
if "viewer_server" in globals():
    viewer_server.close()
viewer_server = LocalViewerServer(archives)
if 2 in visible:
    atlas_widget = show_atlas(visible[2], viewer_server)
    display(atlas_widget)
if 3 in visible:
    display(show_3d(visible[3], viewer_server))


## Inspect original tensors and all accompanying images
Set `RECORD_ID` explicitly, or select records in Atlas and rerun this cell. With no selection the first visible record is used. Tensor details retain their original shape/dtype; `original_tensors` contains the actual tensors for further analysis.


In [ ]:
RECORD_ID = None
inspection_frame = atlas_widget.selection() if 2 in visible else visible[3]
if inspection_frame.empty and RECORD_ID is None:
    raise ValueError("Select a record in Atlas or set RECORD_ID explicitly")
record_id = RECORD_ID or inspection_frame.iloc[0].record_id
record, original_tensors = inspect_record(archives, record_id)
print({key: {"shape": tuple(tensor.shape), "dtype": str(tensor.dtype)}
       for key, tensor in original_tensors.items()})
print(record)
display(show_gallery(projections[dimensions[0]], viewer_server, record_id))


## Finish
Set `CLOSE_VIEWERS=True` and run the final cell when finished to stop the local server. Images and 3D views need it while open. Original archives and projection caches stay on disk. The synthetic example lives in a temporary directory printed above; remove it when no longer needed.


In [ ]:
CLOSE_VIEWERS = False  # Set True when finished inspecting the rendered views.
if CLOSE_VIEWERS:
    viewer_server.close()
